# 01 — Signal Analysis
// Raw signal exploration for UrbanSound8K

**Objective**: Characterize the raw audio signals across all 10 classes.

Covers:
- Time-domain analysis (waveforms, amplitude stats, ZCR)
- Frequency-domain analysis (FFT, PSD, dominant frequencies)
- Time-frequency analysis (STFT, mel spectrograms, window comparison)
- **Wavelet analysis (CWT scalogram, DWT energy decomposition, STFT vs CWT)**
- Noise analysis (SNR estimation)
- Stationarity analysis
- Spectral leakage comparison

In [ ]:
import os, sys
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, ".")


import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import librosa
import librosa.display
import pywt

import config
from src.data_loader import load_metadata, load_audio, get_file_path
from src.signal_analysis import (
    compute_amplitude_stats, compute_zcr,
    compute_fft, compute_psd, find_dominant_frequencies, compute_bandwidth,
    compute_stft, compute_mel_spectrogram, compare_window_sizes,
    compute_cwt, compute_dwt, compute_dwt_energy, compare_stft_vs_cwt,
    estimate_snr, compute_spectral_leakage, check_stationarity
)
from src.visualization import (
    plot_waveforms_per_class, plot_fft_per_class, plot_psd_per_class, plot_spectrogram
)

plt.style.use('dark_background')
%matplotlib inline

## 1. Load Dataset Metadata

In [ ]:
metadata = load_metadata()
print(f"Total samples: {len(metadata)}")
print(f"Classes: {metadata['class'].unique()}")
print(f"Folds: {sorted(metadata['fold'].unique())}")
metadata.head()

## 2. Class Distribution

In [ ]:
class_counts = metadata['class'].value_counts()

fig, ax = plt.subplots(figsize=(10, 5))
fig.patch.set_facecolor('#0a0a0a')
ax.set_facecolor('#111111')
class_counts.plot(kind='bar', color='#00ff41', ax=ax)
ax.set_title('Class Distribution — UrbanSound8K', color='#00ffff')
ax.set_ylabel('Count', color='#888888')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

## 3. Load Sample Audio (1 per class)

In [ ]:
samples = {}
for cls_name in config.CLASS_NAMES:
    row = metadata[metadata['class'] == cls_name].iloc[0]
    file_path = get_file_path(row)
    y = load_audio(file_path)
    samples[cls_name] = y
    print(f"Loaded {cls_name}: {len(y)} samples")

## 4. Time-Domain Analysis

In [ ]:
# Waveforms per class
fig = plot_waveforms_per_class(samples)
plt.show()

# Amplitude statistics
for cls_name, y in samples.items():
    stats = compute_amplitude_stats(y)
    print(f"{cls_name}: RMS={stats['rms']:.4f}, Peak={stats['peak']:.4f}, Crest={stats['crest_factor']:.2f}")

## 5. Frequency-Domain Analysis

In [ ]:
# FFT per class
fft_data = {}
for cls_name, y in samples.items():
    freqs, _, mag_db = compute_fft(y)
    fft_data[cls_name] = (freqs, mag_db)

fig = plot_fft_per_class(fft_data)
plt.show()

# PSD per class
psd_data = {}
for cls_name, y in samples.items():
    freqs, psd = compute_psd(y)
    psd_data[cls_name] = (freqs, psd)
    dominant = find_dominant_frequencies(freqs, psd, n_peaks=3)
    print(f"{cls_name} — Dominant freqs: {[f'{f:.0f} Hz' for f, _ in dominant]}")

fig = plot_psd_per_class(psd_data)
plt.show()

## 6. Time-Frequency Analysis

In [ ]:
# Mel spectrograms for each class
for cls_name, y in samples.items():
    S_db = compute_mel_spectrogram(y)
    plot_spectrogram(S_db, title=f'Mel Spectrogram — {cls_name}',
                     save_name=f'fig_mel_{cls_name}.png')
    plt.show()

In [ ]:
# Window size comparison
example_signal = samples['siren']
window_results = compare_window_sizes(example_signal)

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.patch.set_facecolor('#0a0a0a')

for ax, (size, S_db) in zip(axes.flatten(), window_results.items()):
    librosa.display.specshow(S_db, sr=config.TARGET_SR, hop_length=size//4,
                             x_axis='time', y_axis='hz', ax=ax, cmap='magma')
    ax.set_title(f'Window Size = {size}', color='#ffffff')
    ax.set_facecolor('#111111')

plt.suptitle('Time-Frequency Resolution Trade-off (siren)', color='#00ffff', fontsize=14)
plt.tight_layout()
plt.savefig(f'{config.FIGURES_DIR}/fig_window_size_comparison.png', dpi=150,
            bbox_inches='tight', facecolor='#0a0a0a')
plt.show()

## 6b. Wavelet Transform Analysis

**Wavelet Transform** provides adaptive time-frequency resolution:
- **High frequencies** → fine time resolution, coarse frequency resolution
- **Low frequencies** → coarse time resolution, fine frequency resolution

This is fundamentally different from STFT which uses a **fixed** window size.

We analyze:
1. **CWT (Continuous Wavelet Transform)** — scalogram visualization
2. **DWT (Discrete Wavelet Transform)** — multi-resolution energy decomposition
3. **STFT vs CWT** — time-frequency resolution trade-off comparison

In [ ]:
# --- 6b.1 CWT Scalogram per class ---
# Using Morlet wavelet (good balance of time-frequency localization)

fig, axes = plt.subplots(2, 5, figsize=(20, 8))
fig.patch.set_facecolor('#0a0a0a')

for ax, cls_name in zip(axes.flatten(), config.CLASS_NAMES):
    y = samples[cls_name]
    frequencies, coefficients = compute_cwt(y, config.TARGET_SR)
    
    time_axis = np.arange(len(y)) / config.TARGET_SR
    ax.pcolormesh(time_axis, frequencies, coefficients,
                  cmap='magma', shading='auto')
    ax.set_title(cls_name, color='#ffffff', fontsize=9)
    ax.set_facecolor('#111111')
    ax.set_yscale('log')
    ax.set_ylim(50, 10000)
    ax.tick_params(colors='#888888', labelsize=7)

axes[0, 0].set_ylabel('Frequency (Hz)', color='#888888')
axes[1, 0].set_ylabel('Frequency (Hz)', color='#888888')
for ax in axes[1]:
    ax.set_xlabel('Time (s)', color='#888888', fontsize=8)

plt.suptitle('CWT Scalogram (Morlet) — All Classes', color='#00ffff', fontsize=14)
plt.tight_layout()
plt.savefig(f'{config.FIGURES_DIR}/fig_cwt_scalogram_all_classes.png', dpi=150,
            bbox_inches='tight', facecolor='#0a0a0a')
plt.show()

In [ ]:
# --- 6b.2 DWT Multi-Resolution Energy Decomposition ---
# Daubechies-4 wavelet, showing energy per frequency band

fig, axes = plt.subplots(2, 5, figsize=(20, 7))
fig.patch.set_facecolor('#0a0a0a')

for ax, cls_name in zip(axes.flatten(), config.CLASS_NAMES):
    y = samples[cls_name]
    bands = compute_dwt_energy(y, config.TARGET_SR, wavelet='db4')

    labels = [b['level'] for b in bands]
    ratios = [b['energy_ratio'] * 100 for b in bands]
    freq_labels = [f"{b['freq_range'][0]:.0f}-{b['freq_range'][1]:.0f}" for b in bands]

    colors_bar = plt.cm.magma(np.linspace(0.2, 0.9, len(bands)))
    ax.barh(range(len(bands)), ratios, color=colors_bar, edgecolor='#333')
    ax.set_yticks(range(len(bands)))
    ax.set_yticklabels([f"{l}\n{f} Hz" for l, f in zip(labels, freq_labels)],
                       fontsize=6, color='#cccccc')
    ax.set_title(cls_name, color='#ffffff', fontsize=9)
    ax.set_facecolor('#111111')
    ax.tick_params(colors='#888888', labelsize=7)
    ax.set_xlim(0, 100)

axes[1, 2].set_xlabel('Energy (%)', color='#888888', fontsize=10)
plt.suptitle('DWT Energy Distribution per Frequency Band (db4)', color='#00ffff', fontsize=14)
plt.tight_layout()
plt.savefig(f'{config.FIGURES_DIR}/fig_dwt_energy_per_class.png', dpi=150,
            bbox_inches='tight', facecolor='#0a0a0a')
plt.show()

# Print summary table
print("=== DWT Energy Summary (top band per class) ===")
for cls_name in config.CLASS_NAMES:
    bands = compute_dwt_energy(samples[cls_name], config.TARGET_SR)
    top = max(bands, key=lambda b: b['energy_ratio'])
    print(f"{cls_name:20s} → {top['level']} ({top['freq_range'][0]:.0f}-{top['freq_range'][1]:.0f} Hz): {top['energy_ratio']*100:.1f}%")

In [ ]:
# --- 6b.3 STFT vs CWT — Time-Frequency Resolution Trade-off ---
# Side-by-side comparison on 3 contrasting signals

compare_classes = ['siren', 'gun_shot', 'air_conditioner']

fig, axes = plt.subplots(2, 3, figsize=(18, 8))
fig.patch.set_facecolor('#0a0a0a')

for col, cls_name in enumerate(compare_classes):
    y = samples[cls_name]
    result = compare_stft_vs_cwt(y, config.TARGET_SR)

    # STFT (top row)
    ax_stft = axes[0, col]
    librosa.display.specshow(result['stft'], sr=config.TARGET_SR,
                             hop_length=config.HOP_LENGTH,
                             x_axis='time', y_axis='hz',
                             ax=ax_stft, cmap='magma')
    ax_stft.set_title(f'STFT — {cls_name}', color='#00d4aa', fontsize=11, fontweight='bold')
    ax_stft.set_facecolor('#111111')
    ax_stft.set_ylim(0, 10000)
    ax_stft.tick_params(colors='#888888', labelsize=7)

    # CWT (bottom row)
    ax_cwt = axes[1, col]
    time_axis = np.arange(len(y)) / config.TARGET_SR
    ax_cwt.pcolormesh(time_axis, result['cwt_freqs'], result['cwt_coeffs'],
                      cmap='magma', shading='auto')
    ax_cwt.set_title(f'CWT (Morlet) — {cls_name}', color='#ff6b6b', fontsize=11, fontweight='bold')
    ax_cwt.set_facecolor('#111111')
    ax_cwt.set_yscale('log')
    ax_cwt.set_ylim(50, 10000)
    ax_cwt.set_ylabel('Frequency (Hz)', color='#888888')
    ax_cwt.set_xlabel('Time (s)', color='#888888')
    ax_cwt.tick_params(colors='#888888', labelsize=7)

plt.suptitle('STFT vs CWT — Time-Frequency Resolution Trade-off',
             color='#ffd700', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{config.FIGURES_DIR}/fig_stft_vs_cwt.png', dpi=150,
            bbox_inches='tight', facecolor='#0a0a0a')
plt.show()

print("""
=== STFT vs CWT Trade-off ===

STFT (fixed resolution):
  - Window size determines BOTH time and frequency resolution
  - Good for stationary signals (air_conditioner)
  - Equal resolution at all frequencies

CWT (adaptive resolution):
  - High freq → fine time resolution (good for transients like gun_shot)
  - Low freq → fine frequency resolution (good for tonal sounds like siren)
  - Better for non-stationary signals with mixed time-frequency content
""")

## 7. Noise & Stationarity Analysis

In [ ]:
# SNR estimation per class
print("=== SNR Estimates ===")
for cls_name, y in samples.items():
    snr = estimate_snr(y)
    print(f"{cls_name}: SNR ≈ {snr:.1f} dB")

# Stationarity check
print("\n=== Stationarity Analysis ===")
for cls_name, y in samples.items():
    result = check_stationarity(y)
    label = 'Stationary-like' if result['stationary'] else 'Non-stationary'
    print(f"{cls_name}: {label} (CV_RMS = {result['cv_rms']:.3f})")

## 8. Spectral Leakage Analysis

In [ ]:
# Compare window functions
example = samples['siren']
leakage = compute_spectral_leakage(example)

fig, ax = plt.subplots(figsize=(10, 5))
fig.patch.set_facecolor('#0a0a0a')
ax.set_facecolor('#111111')

colors = {'boxcar': '#ff0080', 'hann': '#00ff41', 'hamming': '#00ffff', 'blackman': '#a855f7'}
for win_name, data in leakage.items():
    ax.plot(data['freqs'][:500], data['magnitude_db'][:500],
            color=colors[win_name], label=win_name, linewidth=1)

ax.set_xlabel('Frequency (Hz)', color='#888888')
ax.set_ylabel('Magnitude (dB)', color='#888888')
ax.set_title('Spectral Leakage — Window Function Comparison', color='#00ffff')
ax.legend()
ax.grid(True, alpha=0.2)
plt.tight_layout()
plt.savefig(f'{config.FIGURES_DIR}/fig_spectral_leakage.png', dpi=150,
            bbox_inches='tight', facecolor='#0a0a0a')
plt.show()